<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در این مثال یک Dataset کوچک برای سیستم HR می‌سازیم. تمام سؤال‌ها و پاسخ‌ها انگلیسی هستند تا با مدل Local شما سازگار باشند.

کد:
1. اسناد HR را تعریف می‌کند.
2. شش نمونه ارزیابی می‌سازد.
3. شناسه اسناد مرتبط را اعتبارسنجی می‌کند.
4. نمونه‌ها را به Development و Test تقسیم می‌کند.
5. Dataset را در قالب JSONL ذخیره می‌کند.

In [1]:
import json
from pathlib import Path

# 1. Knowledge Base
documents = {
    "HR-001": "Employees receive 26 days of annual leave per year.",
    "HR-002": "All annual leave requests require manager approval.",
    "HR-003": "Employees are covered by supplementary health insurance.",
    "HR-004": "Monthly salaries are paid at the end of each month.",
    "HR-005": "Employees can work remotely up to 2 days per week.",
    "HR-006": "Unpaid leave requires approval from the employee's manager."
}

# 2. Evaluation Dataset
evaluation_dataset = [
    {
        "question": "How many annual leave days do employees receive per year?",
        "reference_answer": "Employees receive 26 days of annual leave per year.",
        "relevant_doc_ids": ["HR-001"],
        "question_type": "direct_fact",
        "should_answer": True,
        "split": "development"
    },
    {
        "question": "Who must approve an annual leave request?",
        "reference_answer": "The employee's manager must approve the request.",
        "relevant_doc_ids": ["HR-002"],
        "question_type": "direct_fact",
        "should_answer": True,
        "split": "development"
    },
    {
        "question": "Can I work from home two days a week?",
        "reference_answer": "Yes. Employees can work remotely up to 2 days per week.",
        "relevant_doc_ids": ["HR-005"],
        "question_type": "paraphrased",
        "should_answer": True,
        "split": "development"
    },
    {
        "question": "When are monthly salaries paid?",
        "reference_answer": "Monthly salaries are paid at the end of each month.",
        "relevant_doc_ids": ["HR-004"],
        "question_type": "direct_fact",
        "should_answer": True,
        "split": "test"
    },
    {
        "question": "What is the company's parental leave policy?",
        "reference_answer": "The available documents do not contain this information.",
        "relevant_doc_ids": [],
        "question_type": "unanswerable",
        "should_answer": False,
        "split": "test"
    },
    {
        "question": "What is required to request annual leave, and who approves it?",
        "reference_answer": (
            "Employees receive 26 annual leave days per year, "
            "and all annual leave requests require manager approval."
        ),
        "relevant_doc_ids": ["HR-001", "HR-002"],
        "question_type": "multi_hop",
        "should_answer": True,
        "split": "test"
    }
]

# 3. Validate the dataset
for index, sample in enumerate(evaluation_dataset, start=1):

    # Check that all referenced documents exist
    for doc_id in sample["relevant_doc_ids"]:
        if doc_id not in documents:
            raise ValueError(
                f"Sample {index}: Unknown document ID: {doc_id}"
            )

    # Check answerability label
    if sample["should_answer"] and not sample["relevant_doc_ids"]:
        raise ValueError(
            f"Sample {index}: Answerable question has no relevant documents."
        )

    if not sample["should_answer"] and sample["relevant_doc_ids"]:
        raise ValueError(
            f"Sample {index}: Unanswerable question has relevant documents."
        )

# 4. Split dataset
development_set = [
    item for item in evaluation_dataset
    if item["split"] == "development"
]

test_set = [
    item for item in evaluation_dataset
    if item["split"] == "test"
]

# 5. Save as JSONL
output_path = Path("rag_eval_dataset.jsonl")

with output_path.open("w", encoding="utf-8") as file:
    for sample in evaluation_dataset:
        file.write(
            json.dumps(sample, ensure_ascii=False) + "\n"
        )

# 6. Dataset summary
print(f"Total samples: {len(evaluation_dataset)}")
print(f"Development samples: {len(development_set)}")
print(f"Test samples: {len(test_set)}")
print(f"Dataset saved to: {output_path.resolve()}")

print("\nQuestion types:")
for question_type in sorted({
    item["question_type"] for item in evaluation_dataset
}):
    count = sum(
        item["question_type"] == question_type
        for item in evaluation_dataset
    )
    print(f"{question_type}: {count}")


Total samples: 6
Development samples: 3
Test samples: 3
Dataset saved to: E:\AI-Engineering\4. RAG\notes\7. rag-evaluation\rag_eval_dataset.jsonl

Question types:
direct_fact: 3
multi_hop: 1
paraphrased: 1
unanswerable: 1
